# **HW5 – QAOA and Applications**
_Time required: ~2–3 hours (for students with Qiskit optimization experience)_

**What you’ll practice**
- Formulating combinatorial problems (e.g., MaxCut) as QUBOs
- Building cost and mixer Hamiltonians
- Implementing QAOA circuits in Qiskit
- Variational optimization loops
- Evaluating approximation ratios
- Handling noise in QAOA
- Applications to graph problems in AI/DS

**What to turn in**
- This single notebook (`HW5_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~1.0 or later).
- Use AerSimulator for reproducibility; add noise where specified.
- If stuck, explain reasoning; partial credit for clear work.
- For graphs, use `networkx`; for optimization, use `scipy` or `qiskit.algorithms`.
- Use small graphs (n≤5) to avoid long runtimes.


In [ ]:
# --- Setup (run me first) ---
# Install Qiskit if needed (uncomment in Colab)
# !pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-optimization networkx matplotlib scipy

# Import necessary modules
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit_optimization import QuadraticProgram
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Simulator backend
sim = AerSimulator()

# Function to run circuit and get counts
def get_counts(circ, shots=2000):
    tc = transpile(circ, sim)
    result = sim.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-8):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close:\n{A}\nvs\n{B}")


## Part A — Combinatorial Optimization Basics (≈20 min)

**A1.** Formulate MaxCut for a triangle graph (3 vertices, all connected) as a QUBO.  
**A2.** Draw the graph using networkx; compute classical optimal cut value (should be 1.5 for unit weights).  
**A3.** Short answer: Explain Ising mapping for binary variables; why NP-hard.


In [ ]:
# A1. MaxCut QUBO for triangle
qp = QuadraticProgram('maxcut')
# YOUR CODE HERE: qp.maximize(quadratic={ (0,1):0.5, (0,2):0.5, (1,2):0.5 })
print(qp.prettyprint())

# A2. Graph and optimal
G = nx.Graph()
# YOUR CODE HERE: add_edges_from([(0,1),(0,2),(1,2)])
nx.draw(G, with_labels=True)
plt.show()
cut_value_opt = None  # YOUR CODE HERE: e.g., 1.5 for balanced cut
print("Optimal cut: ", cut_value_opt)

# A3. Written answer: (z=2x-1 maps 0/1 to ±1; quadratic terms penalize/reward; exponential search space)


## Part B — QAOA Components (≈25 min)

**B1.** Build cost Hamiltonian circuit for the triangle (ZZ terms). Draw for γ=π/2.  
**B2.** Add mixer (RX on each qubit) for β=π/4. Evolve from |+++> state; compute statevector.  
**B3.** Short answer: Derive e^{-iγ H_C} action; why mixer promotes exploration.


In [ ]:
# B1. Cost circuit
gamma = np.pi / 2
qc_cost = QuantumCircuit(3)
# YOUR CODE HERE: cx(0,1), rz(2*gamma,1), cx(0,1); repeat for pairs
qc_cost.draw('mpl')
plt.show()

# B2. Mixer and evolve
beta = np.pi / 4
qc_mixer = QuantumCircuit(3)
# YOUR CODE HERE: rx(2*beta, i) for i in 0..2
qc_qaoa_p1 = QuantumCircuit(3)
qc_qaoa_p1.h(range(3))  # |s>
qc_qaoa_p1 = qc_qaoa_p1.compose(qc_cost)
qc_qaoa_p1 = qc_qaoa_p1.compose(qc_mixer)
sv_p1 = Statevector(qc_qaoa_p1)
print("p=1 state: ", sv_p1)

# B3. Written answer: (Phase e^{-iγ z_i z_j /2} for edges; mixer flips bits uniformly, avoiding local minima)


## Part C — Variational Optimization (≈30 min)

**C1.** Define expectation function for <H_C> using shots=2000.  
**C2.** Optimize γ,β for p=1 on triangle using COBYLA; plot landscape if possible.  
**C3.** Short answer: Barren plateaus issue; how p affects approximation.


In [ ]:
# C1. Expectation func
def expectation(params, qc_template, shots=2000):
    gamma, beta = params
    qc = qc_template.copy()
    # YOUR CODE HERE: set gamma, beta in circuit
    qc.measure_all()
    counts = get_counts(qc, shots)
    exp = 0
    # Compute <sum (1-z_i z_j)/2 > over edges
    return -exp  # For minimize

# C2. Optimize
from scipy.optimize import minimize
res = minimize(expectation, [0.5, 0.5], args=(qc_qaoa_p1,), method='COBYLA')
print("Optimal params: ", res.x)
approx_ratio = -res.fun / cut_value_opt
print("Approx ratio: ", approx_ratio)

# C3. Written answer: (Gradients vanish exponentially in n; higher p better approx but more noise/variance)


## Part D — QAOA in Qiskit (≈25 min)

**D1.** Use qiskit.optimization to solve MaxCut on a 4-vertex ring graph with p=2.  
**D2.** Run on simulator; plot histogram of solutions. Compute ratio.  
**D3.** Short answer: Compare to Goemans-Williamson (classical 0.878); NISQ limits.


In [ ]:
# D1. Qiskit QAOA
G_ring = nx.cycle_graph(4)
qp_ring = QuadraticProgram('maxcut_ring')
# YOUR CODE HERE: formulate as in A1
optimizer = COBYLA()
qaoa = QAOA(optimizer=optimizer, reps=2, quantum_instance=sim)
result = qaoa.compute_minimum_eigenvalue(qp_ring.to_ising()[0])
print("QAOA result: ", result)

# D2. Histogram
qc_ring = result.optimal_circuit
qc_ring.measure_all()
counts_ring = get_counts(qc_ring)
plot_histogram(counts_ring)
plt.show()
ratio_ring = result.eigenvalue / 2  # For cycle-4 opt=2
print("Ratio: ", ratio_ring)

# D3. Written answer: (QAOA may beat random but < GW for small p; noise/depth limit large p)


## Part E — Applications & Noise (≈20 min)

**E1.** Short answer: Formulate feature selection as QUBO (max accuracy - λ complexity).  
**E2.** Add depolarizing noise to QAOA sim; compare p=1 vs p=3 fidelity.  
**E3.** Written reflection: QAOA in hybrid ML; future with fault-tolerance.


In [ ]:
# E1. Written answer: (Quadratic terms for feature interactions; linear for individual value)

# E2. Noisy QAOA
from qiskit_aer.noise import NoiseModel, depolarizing_error
noise_model = NoiseModel()
# YOUR CODE HERE: add depol 0.01 to 'rz','rx','cx'
counts_noisy = get_counts(qc_ring, noise_model=noise_model)
plot_histogram(counts_noisy)
plt.show()

# E3. Written answer: (Quantum kernel opt; scalable advantage post-FTQC)
